# Test Notebook

## This Notebok as of use of checking and using all functions for ests of both the functions and data

In [1]:
from constants import *
import os
import numpy as np
from matplotlib import cm
import pandas as pd
import matplotlib.pyplot as plt
from participant_gaze_data_manager import ParticipantGazeDataManager
import matplotlib.animation as animation
import os
import numpy as np
from scipy.ndimage import gaussian_filter
import cv2
import wave
from matplotlib.animation import FFMpegWriter
from constants import *
from moviepy.editor import VideoFileClip, AudioFileClip
from utils import prepare_image_and_gaze
from FixationHandler import FixationHandler
from PanelSymbols import *
from RoiFinder import RoiFinder

In [2]:
p_name = "AG562"
task = "SDMT"
group = "pwMS"  # "HC" or "pwMS"
panel = "0"
panel_path = f'/Volumes/ramot/rotation_students/Noam_M/Results/Behavior/panels_images/SDMT/combined_testable_{panel}.jpg'
data_path = "/Volumes/ramot/rotation_students/Noam_M/Results/Behavior"
output_path = f"/Volumes/ramot/rotation_students/Noam_M/visualized data/test_videos/{p_name}/"
video_path = '/Volumes/ramot/rotation_students/Noam_M/visualized data/test_videos/participant_task_0_AG562.mp4'
img = plt.imread(panel_path)

In [3]:
subject_data = ParticipantGazeDataManager(p_name, data_path, "SDMT", group)


In [5]:
subject_data.matched_data['0'].keys()

dict_keys(['tobii_data', 'task_panel_img', 'messages', 'audio_data', 'strike_score', 'recording_date'])

In [ ]:
from Trial import TrailManager
TM = TrailManager(ParticipantGazeDataManager("AG562", "/Volumes/ramot/rotation_students/Noam_M/Results/Behavior", "SDMT", "pwMS"), 
             '0', 
             plt.imread('/Volumes/ramot/rotation_students/Noam_M/Results/Behavior/panels_images/SDMT/combined_testable_0.jpg'))

Press idx: 2, symbol index: 19, total symbols: 138, total rois: 138, total presses: 69
Trail 2: Symbol 19 at (96, 282), Fixation at (np.float64(158.76274381365096), np.float64(328.598529951913)), Searches: 2, Time: 3486694.0-5386704.0
------------------------------------
Press idx: 3, symbol index: 20, total symbols: 138, total rois: 138, total presses: 69
Trail 3: Symbol 20 at (158, 282), Fixation at (np.float64(229.79463895161948), np.float64(325.1606833934784)), Searches: 2, Time: 4501700.0-6621718.0
------------------------------------
Press idx: 4, symbol index: 21, total symbols: 138, total rois: 138, total presses: 69
Trail 4: Symbol 21 at (220, 282), Fixation at (np.float64(261.0645389556885), np.float64(329.3388372659683)), Searches: 1, Time: 5923374.0-6621718.0
------------------------------------
Press idx: 5, symbol index: 22, total symbols: 138, total rois: 138, total presses: 69
Trail 5: Symbol 22 at (282, 282), Fixation at (np.float64(368.4063850928997), np.float64(324.1

In [5]:
TM.plot_search_to_press_gaps()

AttributeError: 'TrailManager' object has no attribute 'plot_search_to_press_gaps'

In [32]:
res = subject_data.annotate_gaze_events('threshold_based', panel)        
img_resized, res = prepare_image_and_gaze(img, res)

symbols = PanelSymbols.flat_panel_symbols(panel)

roifinder = RoiFinder(panel, img_resized)
roifinder.set_all_attributes(symbols)


fixationhandler = FixationHandler(roifinder)
fixationhandler.set_all_attributes(data = res)



In [33]:
from SearchFinder import SearchFinder, SearchVisualizer

search_finder = SearchFinder(all_fixations= fixationhandler.fixations, roi_finder = roifinder, symbols = symbols)
searches = search_finder.find()


In [34]:
print(searches)

[<SearchFinder.Search object at 0x31d8926d0>, <SearchFinder.Search object at 0x31d891590>, <SearchFinder.Search object at 0x31d4c1c50>, <SearchFinder.Search object at 0x31d942d50>, <SearchFinder.Search object at 0x17a574cd0>, <SearchFinder.Search object at 0x17a576910>, <SearchFinder.Search object at 0x17a576750>, <SearchFinder.Search object at 0x17a5770d0>, <SearchFinder.Search object at 0x17a5762d0>, <SearchFinder.Search object at 0x17a574250>, <SearchFinder.Search object at 0x17a575510>, <SearchFinder.Search object at 0x17a575090>, <SearchFinder.Search object at 0x17a575dd0>, <SearchFinder.Search object at 0x17a576710>, <SearchFinder.Search object at 0x17a576610>, <SearchFinder.Search object at 0x17a575610>, <SearchFinder.Search object at 0x17a576a50>, <SearchFinder.Search object at 0x17a576c10>, <SearchFinder.Search object at 0x17a5758d0>, <SearchFinder.Search object at 0x17a574b10>, <SearchFinder.Search object at 0x17a574a10>, <SearchFinder.Search object at 0x17a577290>, <SearchFi

In [41]:
import cv2

cap = cv2.VideoCapture(video_path)
print("Actual FPS stored in file:", cap.get(cv2.CAP_PROP_FPS))
print("Frame count in file:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("Duration according to file:", cap.get(cv2.CAP_PROP_FRAME_COUNT)/cap.get(cv2.CAP_PROP_FPS))
cap.release()

Actual FPS stored in file: 600.0
Frame count in file: 54630.0
Duration according to file: 91.05


In [36]:
search_visualizer.combine_video_with_graph(video_path, os.path.join(output_path, f"{p_name}_{panel}_combined_video.mp4"))

✅ Combined video saved to /Volumes/ramot/rotation_students/Noam_M/visualized data/test_videos/AG562/AG562_0_combined_video.mp4


In [66]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def debug_fixation_overlay_with_lines(img, gaze_df, searches, search_idx=0):
    search = searches[search_idx]
    fixation_to_plt = [search.fixation_before_search] + search.fixations if search.fixation_before_search else search.fixations
    x_positions = [fix.position[0] for fix in fixation_to_plt]
    y_positions = [fix.position[1] for fix in fixation_to_plt]

    # Global min/max
    x_min, x_max = gaze_df[FIXATION_CSV_KEY_EYE_H].min(), gaze_df[FIXATION_CSV_KEY_EYE_H].max()
    y_min, y_max = gaze_df[FIXATION_CSV_KEY_EYE_V].min(), gaze_df[FIXATION_CSV_KEY_EYE_V].max()

    print(f"X range: {x_min:.1f} - {x_max:.1f}")
    print(f"Y range: {y_min:.1f} - {y_max:.1f}")

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img, aspect='equal')
    ax.scatter(x_positions, y_positions, color='red', s=50, label="Fixations")

    # Draw arrows
    for i in range(len(fixation_to_plt) - 1):
        ax.arrow(
            x_positions[i], y_positions[i],
            x_positions[i+1] - x_positions[i],
            y_positions[i+1] - y_positions[i],
            color='blue', head_width=5, head_length=10, linewidth=1.5, alpha=0.8
        )

    # Bounding box
    rect = patches.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                            linewidth=3, edgecolor='purple', facecolor='none', linestyle='--', label="Global Box")
    ax.add_patch(rect)

    # Explicit lines
    ax.axvline(x_min, color='purple', linestyle='-', linewidth=2, label="x_min")
    ax.axvline(x_max, color='purple', linestyle='-', linewidth=2, label="x_max")
    ax.axhline(y_min, color='green', linestyle='-', linewidth=2, label="y_min")
    ax.axhline(y_max, color='green', linestyle='-', linewidth=2, label="y_max")

    # Flip y-axis to match image coordinate system
    ax.set_xlim([0, img.shape[1]])
    ax.set_ylim([img.shape[0], 0])

    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
debug_fixation_overlay_with_lines(img_resized,res, searches)

In [36]:
from MCAnalyzer import build_symbol_transition_graph
G = build_symbol_transition_graph(searches)

print("Nodes:", G.nodes())
print("Edges with weights:")
for u, v, data in G.edges(data=True):
    print(f"{u} → {v}: {data['weight']:.3f}")

ValueError: None cannot be a node

ValueError: None cannot be a node